# Transcript-Prompted Whisper for Sw↔En Speech Translation

End-to-end notebook for the lexicon-augmented transcript-prompted ST experiment.

**Pipeline:**
1. Build bilingual lexicon from existing alignments (sw2en + en2sw)
2. Build FLEURS sw_ke test reference set
3. Sanity-check KenSpeech indexing
4. Train transcript-prompted Whisper-small (~6 hrs T4)
5. Generate predictions for 3 system variants
6. Evaluate BLEU + chrF
7. Persist results

**Each section can be run independently** — `if not os.path.exists(...)` guards skip completed steps.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy sentencepiece sacrebleu "datasets<4.0"

In [ ]:
# HuggingFace authentication (for FLEURS download + optional model push)
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HF login successful')
except Exception as e:
    print(f'HF token not set ({e}). FLEURS download may rate-limit.')

In [ ]:
# Paths and config
import os, sys, json, torch
from pathlib import Path

REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

# Inputs (saved Kaggle datasets)
KENSPEECH_DIR = '/kaggle/input/kenspeech-sw'
SW2EN_DIR = '/kaggle/input/datasets/victormugambi/mambosw2en'
EN2SW_DIR = '/kaggle/input/datasets/victormugambi/mamboen2sw'

# Working dirs
WORK_DIR = '/kaggle/working'
LEXICON_PATH = f'{WORK_DIR}/lexicon.jsonl'
FLEURS_TEST_DIR = f'{WORK_DIR}/fleurs_sw_ke_test'
MODEL_OUT_DIR = f'{WORK_DIR}/whisper_st_sw2en'
PRED_DIR = f'{WORK_DIR}/predictions'
os.makedirs(PRED_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"GPU count: {torch.cuda.device_count()}")
for path, label in [(KENSPEECH_DIR, 'KenSpeech'), (SW2EN_DIR, 'sw2en alignments'), (EN2SW_DIR, 'en2sw alignments'), (REPO_DIR, 'Repo')]:
    print(f"  {label:25s}: {'OK' if os.path.exists(path) else 'MISSING'}  {path}")

## Step 1: Build Bilingual Lexicon from Alignments

Uses both directions' alignments for cross-validation. Bidirectional pairs are flagged as high-confidence.

In [ ]:
# Verify alignment paths first
for label, path in [('sw2en', f'{SW2EN_DIR}/alignments/sw2en'), ('en2sw', f'{EN2SW_DIR}/alignments/en2sw')]:
    if os.path.isdir(path):
        n = len([f for f in os.listdir(path) if f.endswith('.json') and f != 'index.jsonl'])
        print(f'  [{label}] OK  -- {n} alignment files at {path}')
    else:
        print(f'  [{label}] MISSING  -- {path}')
        # Try to find alignments dir under the dataset root
        ds_root = path.split('/alignments/')[0]
        if os.path.isdir(ds_root):
            print(f'    Contents of {ds_root}:')
            for entry in sorted(os.listdir(ds_root))[:20]:
                print(f'      {entry}')

In [ ]:
# Build the lexicon (alignments + Wikidata)
if os.path.exists(LEXICON_PATH) and os.path.getsize(LEXICON_PATH) > 0:
    n = sum(1 for _ in open(LEXICON_PATH))
    print(f'Lexicon already exists with {n} entries -- skipping rebuild. Delete to regenerate.')
else:
    !cd {REPO_DIR} && python whisper_st/build_lexicon.py \
        --alignments sw2en={SW2EN_DIR}/alignments/sw2en \
                     en2sw={EN2SW_DIR}/alignments/en2sw \
        --output_path {LEXICON_PATH} \
        --min_freq 2 \
        --min_alignment_consistency 0.4 \
        --include_wikidata

In [ ]:
# Sanity-check the lexicon
n_total = 0
n_bidir = 0
examples = []
with open(LEXICON_PATH) as f:
    for line in f:
        e = json.loads(line)
        n_total += 1
        if e.get('bidirectional'):
            n_bidir += 1
            if len(examples) < 15:
                examples.append(e)
print(f'Total entries: {n_total} | Bidirectional: {n_bidir}')
print('\nTop 15 bidirectional entries (high-confidence):')
for e in examples:
    print(f"  {e['sw']:25s} -> {e['en']:30s} (freq={e['freq']}, consistency={e.get('consistency')})")

## Step 2: Build FLEURS sw_ke Test Reference Set

FLEURS is parallel across languages by `id`, so we use the `sw_ke` test split for audio and the `en_us` test split for English references.

In [ ]:
if os.path.exists(f'{FLEURS_TEST_DIR}/refs.jsonl') and os.path.getsize(f'{FLEURS_TEST_DIR}/refs.jsonl') > 0:
    n = sum(1 for _ in open(f'{FLEURS_TEST_DIR}/refs.jsonl'))
    n_wavs = len(list(Path(f'{FLEURS_TEST_DIR}/audio').glob('*.wav'))) if os.path.exists(f'{FLEURS_TEST_DIR}/audio') else 0
    print(f'FLEURS test set already built: {n} refs, {n_wavs} wavs -- skipping')
else:
    from datasets import load_dataset
    import soundfile as sf, numpy as np

    os.makedirs(f'{FLEURS_TEST_DIR}/audio', exist_ok=True)

    print('Downloading FLEURS sw_ke test split...')
    sw_ds = load_dataset('google/fleurs', 'sw_ke', split='test', trust_remote_code=True)
    print(f'  sw_ke test: {len(sw_ds)} samples')

    print('Downloading FLEURS en_us test split (for parallel English references)...')
    en_ds = load_dataset('google/fleurs', 'en_us', split='test', trust_remote_code=True)
    en_by_id = {s['id']: s['transcription'] for s in en_ds}
    print(f'  en_us test: {len(en_by_id)} samples')

    # Write parallel pairs (only those with matching IDs)
    n_paired = 0
    with open(f'{FLEURS_TEST_DIR}/refs.jsonl', 'w', encoding='utf-8') as f:
        for i, sample in enumerate(sw_ds):
            sample_id = sample['id']
            if sample_id not in en_by_id:
                continue
            wav_name = f'fleurs_sw_ke_{i:05d}.wav'
            audio = sample['audio']
            sf.write(f'{FLEURS_TEST_DIR}/audio/{wav_name}', np.asarray(audio['array'], dtype=np.float32), audio['sampling_rate'])
            f.write(json.dumps({
                'audio': wav_name,
                'reference_en': en_by_id[sample_id],
                'reference_sw': sample['transcription'],
                'id': sample_id,
            }, ensure_ascii=False) + '\n')
            n_paired += 1
    print(f'\nWrote {n_paired} parallel test pairs to {FLEURS_TEST_DIR}/')

## Step 3: Sanity-Check KenSpeech Indexing

Confirms `sample_idx` in translation JSONs matches the order returned by `KenSpeechLoader.__getitem__`. If MISMATCH, the dataset class needs a small patch.

In [ ]:
from data.prepare.kenspeech_loader import KenSpeechLoader
ks = KenSpeechLoader(load_audio=False, local_dir=KENSPEECH_DIR)

# Use the saved sw2en translations dataset (same source files as alignments)
trans_root = None
for cand in [f'{SW2EN_DIR}/translations/sw2en', f'{SW2EN_DIR}/translations']:
    if os.path.isdir(cand):
        trans_root = cand
        break
if trans_root is None:
    raise RuntimeError(f'Could not find translations under {SW2EN_DIR}. Inspect contents.')

files = sorted(Path(trans_root).glob('*.json'))
print(f'Translations root: {trans_root}, {len(files)} files')

# Check 5 random samples
import random
random.seed(0)
n_match, n_mismatch = 0, 0
for jp in random.sample(files, min(5, len(files))):
    data = json.load(open(jp))
    si = data['sample_idx']
    sw_translation = data['source_text'].strip()
    sw_kenspeech = ks[si]['sentence'].strip()
    match = sw_translation == sw_kenspeech
    print(f'  sample_idx={si}: {"MATCH" if match else "MISMATCH"}')
    print(f'    translation src: {sw_translation[:80]}')
    print(f'    kenspeech[{si}]: {sw_kenspeech[:80]}')
    n_match += int(match); n_mismatch += int(not match)

if n_mismatch > 0:
    print(f'\nWARNING: {n_mismatch}/{n_match+n_mismatch} mismatches. Dataset class needs sample_idx remapping patch.')
else:
    print(f'\nOK: all {n_match} samples match. Training can proceed.')

## Step 4: Restore Translations Locally + Train Whisper-small

Trainer reads from local working dir, so we copy translations from the saved Kaggle dataset.

**Expected runtime: ~6 hrs on T4** (Whisper-small, 3 epochs, batch 8, ~5800 samples).

In [ ]:
import shutil
TRANSLATION_DIR = f'/kaggle/working/hibiki-sw/translations/sw2en'
if not os.path.isdir(TRANSLATION_DIR) or not os.listdir(TRANSLATION_DIR):
    src = trans_root  # discovered in Step 3
    os.makedirs(TRANSLATION_DIR, exist_ok=True)
    shutil.copytree(src, TRANSLATION_DIR, dirs_exist_ok=True)
n = len(list(Path(TRANSLATION_DIR).glob('*.json')))
print(f'Translations ready locally: {n} files in {TRANSLATION_DIR}')

In [ ]:
# Train transcript-prompted Whisper-small
if os.path.exists(f'{MODEL_OUT_DIR}/final/config.json'):
    print(f'Trained model already at {MODEL_OUT_DIR}/final -- skipping training. Delete to retrain.')
else:
    !cd {REPO_DIR} && python whisper_st/train.py \
        --base_model openai/whisper-small \
        --translations_dir {TRANSLATION_DIR} \
        --kenspeech_dir {KENSPEECH_DIR} \
        --output_dir {MODEL_OUT_DIR} \
        --batch_size 8 --grad_accum 2 \
        --lr 1e-5 --epochs 3 --warmup_steps 200 \
        --ctc_loss_weight 0.3 \
        --num_workers 2

## Step 5: Generate Predictions for 3 System Variants

1. **Vanilla** — Whisper-small zero-shot ST baseline (no fine-tuning, no lexicon)
2. **Fine-tuned** — our model, transcript-prompted, no lexicon
3. **Fine-tuned + Lexicon** — our model + lexicon-augmented prompts

Each takes ~15-30 min for ~250-300 FLEURS test wavs on T4.

In [ ]:
# Variant 1: vanilla Whisper-small zero-shot Sw->En
PRED_VANILLA = f'{PRED_DIR}/preds_vanilla.jsonl'
if os.path.exists(PRED_VANILLA) and os.path.getsize(PRED_VANILLA) > 0:
    print(f'Vanilla predictions exist at {PRED_VANILLA} -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/baseline_vanilla.py \
        --base_model openai/whisper-small \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_VANILLA}

In [ ]:
# Optional: stronger vanilla baseline with Whisper-large-v3 sharded across 2x T4
# (Skip if you only have one GPU or want to save time.)
PRED_VANILLA_LARGE = f'{PRED_DIR}/preds_vanilla_largev3.jsonl'
RUN_LARGE_BASELINE = False  # toggle on if you have 2x T4 and want a stronger baseline
if RUN_LARGE_BASELINE:
    if os.path.exists(PRED_VANILLA_LARGE):
        print(f'Whisper-large-v3 baseline exists at {PRED_VANILLA_LARGE} -- skipping')
    else:
        !cd {REPO_DIR} && python whisper_st/baseline_vanilla.py \
            --base_model openai/whisper-large-v3 \
            --audio_dir {FLEURS_TEST_DIR}/audio \
            --output_path {PRED_VANILLA_LARGE} \
            --device_map auto

In [ ]:
# Variant 2: our fine-tuned transcript-prompted model (no lexicon)
PRED_FT = f'{PRED_DIR}/preds_finetuned.jsonl'
if os.path.exists(PRED_FT) and os.path.getsize(PRED_FT) > 0:
    print(f'Fine-tuned predictions exist at {PRED_FT} -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_OUT_DIR}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_FT}

In [ ]:
# Variant 3: our fine-tuned model + lexicon-augmented prompts
PRED_FT_LEX = f'{PRED_DIR}/preds_finetuned_lex.jsonl'
if os.path.exists(PRED_FT_LEX) and os.path.getsize(PRED_FT_LEX) > 0:
    print(f'Fine-tuned+lex predictions exist at {PRED_FT_LEX} -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_OUT_DIR}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_FT_LEX} \
        --lexicon_path {LEXICON_PATH}

## Step 6: Evaluate BLEU + chrF on All Variants

In [ ]:
# Build the predictions argument list dynamically (skip variants whose files don't exist)
variants = []
for name, path in [
    ('vanilla',         PRED_VANILLA),
    ('vanilla_largev3', PRED_VANILLA_LARGE),
    ('finetuned',       PRED_FT),
    ('finetuned_lex',   PRED_FT_LEX),
]:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        variants.append(f'{name}={path}')
print('Variants found:', variants)

preds_args = ' '.join(variants)
!cd {REPO_DIR} && python whisper_st/eval_bleu.py \
    --references_path {FLEURS_TEST_DIR}/refs.jsonl \
    --predictions {preds_args} \
    --show_examples 12

## Step 7: Persist Results

Save the model + predictions + lexicon to a Kaggle dataset version (or push to HuggingFace) so the work isn't lost when the session ends.

In [ ]:
# Show what's in /kaggle/working that should be saved
print('=== Output Summary ===')
for name, path in [
    ('Lexicon',           LEXICON_PATH),
    ('FLEURS test refs',  f'{FLEURS_TEST_DIR}/refs.jsonl'),
    ('Trained model',     f'{MODEL_OUT_DIR}/final'),
    ('Predictions',       PRED_DIR),
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            n = sum(1 for _ in Path(path).rglob('*') if _.is_file())
            size_mb = sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file()) / 1e6
            print(f'  {name:25s}: {n} files, {size_mb:.1f} MB at {path}')
        else:
            size_mb = os.path.getsize(path) / 1e6
            print(f'  {name:25s}: {size_mb:.2f} MB at {path}')
    else:
        print(f'  {name:25s}: [NOT FOUND] {path}')

print('\n=== To Persist ===')
print('Option A: click "Save Version" in the Kaggle UI (top-right) -- saves all of /kaggle/working')
print('Option B (HF push): set HF_REPO and uncomment the cell below')

In [ ]:
# Optional: push to HuggingFace Hub
# from huggingface_hub import HfApi
# HF_REPO = 'victormugambi/whisper-st-sw2en'  # change to your repo
# api = HfApi(token=os.environ.get('HF_TOKEN'))
# api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True, private=True)
# api.upload_folder(folder_path=f'{MODEL_OUT_DIR}/final', repo_id=HF_REPO, repo_type='model',
#                   commit_message='Transcript-prompted Whisper-small for Sw->En ST')
# print(f'Pushed to https://huggingface.co/{HF_REPO}')